---
# Post-TARE Model Run: Timeseries Peak Load Analysis (AWS / BuildStockQuery)
---

**Author:** Jordan M. Joseph, PhD — Carnegie Mellon University

Queries the AWS-hosted ResStock EUSS 2022.1.1 timeseries database via BuildStockQuery
to compute **county-level peak load changes** under two adoption scenarios:
- **(a) 100% adoption counterfactual** — all filtered building IDs adopt (upper bound)
- **(b) Economically-constrained adoption** — only Tier 1 + Tier 2 adopters (TARE output)

**Prerequisite:** Steps 0–0c must be run first (TARE data loaded, MP selected).

**Dataset scope:** ResStock 2022.1.1 (EUSS, AMY2018). ResStock 2025.1 is future work.

**Primary test case:** Allegheny County, PA (FIPS 42003) — validate here before scaling.

See methodology notes in `peak_load_methodology.md`.

---
## Step 0: Imports and Configuration
---

In [ ]:
import os
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from config import PROJECT_ROOT
from cmu_tare_model.constants import (
    ALLOWED_HOUSING_TYPES,
    VALID_MENU_MPS,
    VERBOSE,
    REMDB_COST_SCENARIO_KEYS,
    RCM_MODELS,
    PRIVATE_DISCOUNT_RATE_SHORT_KEYS,
)

from cmu_tare_model.utils.column_names import (
    create_npv_col,
    create_capital_col,
)

from cmu_tare_model.utils.load_exported_results_to_df import load_measure_package_data

from cmu_tare_model.adoption_kpis.kpi_functions import (
    mp_to_upgrade,
    load_euss_baseline,
    load_euss_upgrade,
    calculate_price_ratios,
    compute_thermal_cop_by_state,
    compute_spark_gap_metrics,
    compute_scenario_demand,
    aggregate_demand_by_state,
    FUEL_PRICES_PATH,
    SHAPEFILE_PATH,
    HEATING_FUEL_COLS,
    HP_BACKUP_ELEC_COL,
    HP_FANS_PUMPS_COL,
)
from cmu_tare_model.adoption_kpis.visualize_geospatial_data import (
    prepare_state_geodataframe,
    create_choropleth_map,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)

print("✓ Imports loaded")

---
## Step 0b: Measure Package Selection
---

In [ ]:
SELECTABLE_MPS = [mp for mp in VALID_MENU_MPS if mp != 0]

try:
    _ = input_measure_package
    batch_mode = True
    selected_mps = [int(input_measure_package)]
    print(f"BATCH MODE: Running for MP{selected_mps[0]}")
except NameError:
    batch_mode = False
    print(f"Available measure packages: {SELECTABLE_MPS}")
    mp_input = input("Enter MP numbers (comma-separated, or 'all'): ").strip()
    if mp_input.lower() == 'all':
        selected_mps = SELECTABLE_MPS
    else:
        selected_mps = [int(x.strip()) for x in mp_input.split(',') if x.strip().isdigit()]
        selected_mps = [mp for mp in selected_mps if mp in SELECTABLE_MPS]
    if not selected_mps:
        selected_mps = [4]
        print("No valid MPs selected. Defaulting to MP4.")

print(f"\nSelected measure packages: {selected_mps}")

---
## Step 0c: Load TARE Model Data (Measure Packages 3, 4)
---

Load pre-computed TARE model outputs for the selected measure packages.
Required for Step 4d (Private NPV extraction).

If the top-section data loading cells have already been run, this will reuse `DATAFRAMES_BY_MP`.

In [ ]:
# =============================================================================
# STEP 0c: LOAD TARE MODEL DATA (for NPV extraction)
# =============================================================================
# Check if DATAFRAMES_BY_MP was already loaded by the top-section cells.
# If not, prompt for the output folder and load for selected MPs.

try:
    _ = DATAFRAMES_BY_MP
    print(f"DATAFRAMES_BY_MP already loaded: {list(DATAFRAMES_BY_MP.keys())}")
except NameError:
    print("DATAFRAMES_BY_MP not found — loading TARE model outputs...")

    # Check if output_folder_path is already defined (from top section)
    try:
        _ = output_folder_path
        print(f"  Using existing output_folder_path: {output_folder_path}")
    except NameError:
        output_folder_path = os.path.join(PROJECT_ROOT, "cmu_tare_model", "output_results")
        location_id = input("Enter location ID (e.g., 'National' or 'PA'): ").strip()
        model_run_date_time = input("Enter model run timestamp (YYYY-MM-DD_HH-MM): ").strip()
        print(f"  output_folder_path: {output_folder_path}")
        print(f"  location_id: {location_id}")
        print(f"  model_run_date_time: {model_run_date_time}")

    DATAFRAMES_BY_MP = {}
    for mp in selected_mps:
        DATAFRAMES_BY_MP[mp] = load_measure_package_data(
            mp, output_folder_path, location_id, model_run_date_time
        )

    print(f"\n✓ Loaded TARE data for MPs: {list(DATAFRAMES_BY_MP.keys())}")

---
## Step 1: BuildStockQuery Installation and AWS Configuration
---

Installs BuildStockQuery (NREL SWR-23-58) and verifies AWS credentials are configured.
BuildStockQuery connects to the OEDI data lake via AWS Athena and returns results
as pandas DataFrames.

**Required before running:**
- AWS CLI configured (`aws configure`) with credentials that have Athena + S3 read access
- BuildStockQuery installed in the `cmu-tare-model` conda environment

**References:**
- Docs: https://github.com/NREL/buildstock-query/wiki
- Examples: https://github.com/NREL/buildstock-query/tree/main/example_usage
- OEDI S3 browser: https://data.openei.org/s3_viewer?bucket=oedi-data-lake&prefix=nrel-pds-building-stock%2F

In [ ]:
# CONTEXT: Setting up BuildStockQuery for ResStock EUSS 2022.1.1 timeseries queries
# via AWS Athena on the OEDI data lake. Python 3.11, conda env: cmu-tare-model.
#
# TASK: 
#   1. Check if buildstock_query is importable; if not, print install instructions.
#   2. Check AWS credentials are configured (boto3 STS get_caller_identity).
#   3. Print the confirmed AWS region and identity as a sanity check.
#   4. Import ResStockQuery (or equivalent BuildStockQuery entry point).
#
# INPUTS: None (environment check only)
# OUTPUTS: Printed confirmation. `bsq_available` (bool). `ResStockQuery` class imported.
# CONSTRAINTS: Type hints on any helper functions. Fail fast with a clear error message
#              if AWS credentials are missing. Do not hardcode any credentials.

# --- PLACEHOLDER: Opus/Copilot drafts implementation below ---

---
## Step 2: OEDI Table Discovery — Confirm EUSS 2022.1.1 Schema
---

Before writing any queries, we need to confirm the exact Athena database name,
table names, and column schema for ResStock EUSS 2022.1.1 timeseries data.

**Key unknowns to resolve in this step:**
- What is the Athena database name for EUSS 2022.1.1?
- What are the timeseries table names (baseline = MP0, upgrade = MP3/MP4)?
- What county-level geography columns exist? (FIPS code? GISJOIN? county name string?)
- What is the exact column name and format for `bldg_id` in the timeseries table?
- What are the electricity end-use column names for total consumption per hour?
- What is the `timestamp` column name and format (ISO string? epoch integer? hour index 1–8760?)

**OEDI S3 path (confirm via browser):**
`s3://oedi-data-lake/nrel-pds-building-stock/end-use-load-profiles-for-us-building-stock/2022/resstock_amy2018_release_1.1/`

**Expected output of this step:** A printed schema summary that confirms all column
names needed for Steps 3–6.

In [1]:
# CONTEXT: We need to discover the Athena schema for ResStock EUSS 2022.1.1 before
# querying timeseries data. The data is on the OEDI data lake (AWS Athena).
# BuildStockQuery or direct boto3/Athena calls can be used to inspect table schemas.
#
# TASK:
#   1. Initialize a BuildStockQuery ResStockQuery object pointing to EUSS 2022.1.1.
#      Use the correct Athena database name and S3 output location.
#   2. List available tables in the database.
#   3. For the baseline timeseries table, print: column names, dtypes, and a 3-row sample.
#   4. Identify and print: the bldg_id column, timestamp column, county geography
#      column(s) (FIPS or GISJOIN), and total electricity consumption column(s).
#   5. Store confirmed column names as constants for use in later steps.
#
# INPUTS: AWS credentials (from environment). BuildStockQuery installed.
# OUTPUTS:
#   - `rsq` (ResStockQuery): initialized query object
#   - `BLDG_ID_COL` (str): confirmed bldg_id column name
#   - `TIMESTAMP_COL` (str): confirmed timestamp/hour column name
#   - `COUNTY_COL` (str): confirmed county geography column name
#   - `ELEC_TOTAL_COL` (str): confirmed total electricity column name
# CONSTRAINTS: Type hints on any helper functions. Print schema clearly.
#              If a column name is uncertain, print all available columns and flag for review.

# --- PLACEHOLDER: Opus/Copilot drafts implementation below ---

# Confirmed column names (fill in after running discovery):
BLDG_ID_COL    = "bldg_id"          # TODO: confirm
TIMESTAMP_COL  = "timestamp"        # TODO: confirm (may be 'hour' or integer index)
COUNTY_COL     = "in.county"        # TODO: confirm (FIPS? GISJOIN? string name?)
ELEC_TOTAL_COL = "electricity_kwh"  # TODO: confirm

---
## Step 3: County Geography Mapping — FIPS / GISJOIN to Shapefile
---

ResStock EUSS uses either FIPS codes or GISJOIN identifiers for county-level geography.
We need to confirm which identifier is present and map it to standard Census TIGER/Line
shapefiles for choropleth visualization.

**GISJOIN format:** `G` + state FIPS (2 digits, zero-padded) + `0` + county FIPS
(3 digits, zero-padded). Example: Allegheny County PA = `G4200030`.

**Standard FIPS format:** 5-digit string. Example: Allegheny County PA = `42003`.

**Goal of this step:** Build a lookup table `county_geo_df` that maps ResStock's
county identifier → FIPS → county name → state, so all downstream results can be
joined to shapefiles.

**Test case:** Confirm that Allegheny County, PA appears with FIPS `42003`.

In [2]:
# CONTEXT: ResStock EUSS 2022.1.1 uses county-level geography identifiers (FIPS or
# GISJOIN) to tag each building. We need to map these to standard Census TIGER/Line
# county shapefiles for visualization. Python 3.11, geopandas already imported.
#
# TASK:
#   1. Query the ResStock metadata table (annual file or Athena) to get all unique
#      county geography values from `COUNTY_COL`.
#   2. Determine the format: is it FIPS (5-digit int or string) or GISJOIN (G+FIPS)?
#   3. Build a mapping DataFrame `county_geo_df` with columns:
#      [resstock_county_id, fips_5digit, county_name, state_abbr]
#   4. Join to the Census TIGER/Line county shapefile at `SHAPEFILE_PATH`.
#   5. Confirm Allegheny County PA (FIPS 42003) is present. Print a sample of 5 rows.
#
# INPUTS:
#   - `rsq` (ResStockQuery): initialized BuildStockQuery object
#   - `COUNTY_COL` (str): confirmed county column name from Step 2
#   - `SHAPEFILE_PATH` (str): path to county-level shapefile (from existing constants)
# OUTPUTS:
#   - `county_geo_df` (pd.DataFrame): mapping table [resstock_county_id → FIPS → name]
#   - `gdf_counties` (gpd.GeoDataFrame): shapefile joined to county_geo_df
# CONSTRAINTS: Type hints. Fail with clear message if Allegheny County is missing.
#              Do not assume GISJOIN vs FIPS — detect from the data.

# --- PLACEHOLDER: Opus/Copilot drafts implementation below ---

---
## Step 4: Extract Adopter Building IDs from TARE Results
---

The TARE model assigns each building (`bldg_id`) to an adoption tier:
- **Tier 1** — Private NPV positive (saves money without incentives)
- **Tier 2** — Private NPV positive only with IRA rebates
- **Tier 3** — Requires public benefits (social cost of carbon) to be justified
- **Tier 4** — No adoption pathway under current conditions

For the **economically-constrained scenario**, adopters = Tier 1 + Tier 2.
For the **100% adoption counterfactual**, adopters = all filtered building IDs.

This step extracts both sets of building IDs, organized by county (FIPS), for use
in the timeseries queries in Steps 5–6.

**Output structure:**
```python
adopter_ids_by_county = {
    "42003": {                     # FIPS for Allegheny County, PA
        "tier1": [101, 204, ...],
        "tier2": [305, 412, ...],
        "constrained": [101, 204, 305, 412, ...],  # tier1 + tier2
        "all_filtered": [101, 204, ..., 99999],    # 100% adoption
    }
}
```

In [3]:
# CONTEXT: TARE model results are stored in DATAFRAMES_BY_MP (loaded in Step 0c).
# Each row is a building (bldg_id index) with columns for private NPV, tier assignment,
# county geography, and applicability. We need to extract adopter building IDs grouped
# by county for both the constrained and 100% adoption scenarios.
#
# TASK:
#   1. For the selected MP (selected_mps[0]), access DATAFRAMES_BY_MP[mp]['fixed_base'].
#   2. Identify the tier assignment column(s). Print available columns if uncertain.
#   3. Extract two sets of bldg_ids per county:
#      (a) `constrained`: Tier 1 + Tier 2 adopters
#      (b) `all_filtered`: all buildings that passed the occupancy/SF/applicable filters
#   4. Build `adopter_ids_by_county` dict keyed by FIPS (from county_geo_df mapping).
#   5. Print a summary: total adopters vs total buildings for the test county (FIPS 42003).
#
# INPUTS:
#   - `DATAFRAMES_BY_MP` (Dict[int, Dict]): TARE results loaded in Step 0c
#   - `selected_mps` (List[int]): MP numbers selected in Step 0b
#   - `county_geo_df` (pd.DataFrame): county mapping from Step 3
# OUTPUTS:
#   - `adopter_ids_by_county` (Dict[str, Dict[str, List[int]]]): 
#     keyed by FIPS → {"tier1", "tier2", "constrained", "all_filtered"}
#   - `primary_mp` (int): the single selected MP (selected_mps[0])
# CONSTRAINTS: Type hints on all helper functions. Google/NumPy docstrings.
#              Fail fast if tier column names are not found — print available columns.

# --- PLACEHOLDER: Opus/Copilot drafts implementation below ---

---
## Step 5: Test Query — Baseline Timeseries for Allegheny County (FIPS 42003)
---

Before building the full pipeline, validate the query pattern on a single county.
Allegheny County, PA (FIPS 42003) is the primary case study.

**What this step produces:**
- `df_ts_baseline_allegheny`: shape (8760 × N_buildings) or pre-aggregated (8760,)
  containing hourly baseline (MP0) electricity consumption for all buildings in county

**Design decision to resolve here:**
- Query strategy A: Pull all individual building timeseries → aggregate locally in pandas
  (higher Athena cost, more flexible locally)
- Query strategy B: Push aggregation into Athena SQL → pull only the hourly sum
  (lower cost, less flexible for adopter/non-adopter split)

**Recommendation:** Start with Strategy A on Allegheny County only to validate the
schema, then decide whether to push aggregation to Athena for the national loop.

In [4]:
# CONTEXT: Testing BuildStockQuery against the OEDI EUSS 2022.1.1 timeseries Athena
# table for a single county (Allegheny County, PA, FIPS 42003) before scaling.
# We want hourly baseline (MP0) electricity consumption for all buildings in this county.
#
# TASK:
#   1. Filter `all_filtered` building IDs for FIPS 42003 from `adopter_ids_by_county`.
#   2. Query the EUSS baseline timeseries table via BuildStockQuery for those bldg_ids.
#      Use `BLDG_ID_COL`, `TIMESTAMP_COL`, `ELEC_TOTAL_COL` confirmed in Step 2.
#   3. Return a DataFrame `df_ts_baseline_allegheny` with columns:
#      [bldg_id, hour (1–8760), baseline_kwh].
#   4. Print: shape, memory usage, min/max hour, min/max kWh. Sample 5 rows.
#   5. Time the query and print elapsed seconds — we'll use this to estimate national cost.
#
# INPUTS:
#   - `rsq` (ResStockQuery): initialized BuildStockQuery object
#   - `adopter_ids_by_county` (dict): from Step 4
#   - `BLDG_ID_COL`, `TIMESTAMP_COL`, `ELEC_TOTAL_COL` (str): from Step 2
# OUTPUTS:
#   - `df_ts_baseline_allegheny` (pd.DataFrame): [bldg_id, hour, baseline_kwh]
#   - `query_time_s` (float): elapsed query time in seconds
# CONSTRAINTS: Type hints. Wrap query in try/except — print Athena error clearly.
#              Do not pull more columns than needed (minimize scan cost).

# --- PLACEHOLDER: Opus/Copilot drafts implementation below ---

---
## Step 6: Test Query — Upgrade Timeseries (MP3 or MP4) for Allegheny County
---

Query the post-retrofit timeseries for the same county and building IDs.
The upgrade table contains electricity consumption after heat pump installation.

**Note:** Only buildings where `applicability == True` have valid upgrade data.
The `adopter_ids_by_county["42003"]["all_filtered"]` list already respects this filter.

**Key column to confirm:** Does the upgrade timeseries table have the same schema
as the baseline table, or are column names different? Resolve this here before
building the aggregation logic in Step 7.

In [ ]:
# CONTEXT: Querying the ResStock EUSS upgrade timeseries table (MP3 or MP4) for
# Allegheny County, PA (FIPS 42003). This is the post-retrofit electricity consumption
# for buildings that receive a heat pump under the selected measure package.
#
# TASK:
#   1. Identify the correct Athena table name for the upgrade scenario (MP3 or MP4).
#      This may differ from the baseline table name — check Athena schema if uncertain.
#   2. Query the upgrade timeseries for bldg_ids in `adopter_ids_by_county["42003"]["all_filtered"]`.
#   3. Return `df_ts_upgrade_allegheny` with columns: [bldg_id, hour, retrofit_kwh].
#   4. Confirm schema matches baseline (same hour range, same bldg_id format).
#   5. Print shape, sample rows, and flag any bldg_ids present in baseline but missing
#      from upgrade (these are non-applicable buildings — should be zero after filter).
#
# INPUTS:
#   - `rsq` (ResStockQuery): initialized BuildStockQuery object
#   - `primary_mp` (int): selected measure package (3 or 4)
#   - `adopter_ids_by_county` (dict): from Step 4
#   - `BLDG_ID_COL`, `TIMESTAMP_COL`, `ELEC_TOTAL_COL` (str): from Step 2
# OUTPUTS:
#   - `df_ts_upgrade_allegheny` (pd.DataFrame): [bldg_id, hour, retrofit_kwh]
# CONSTRAINTS: Type hints. Print a warning if any adopter bldg_ids are missing
#              from the upgrade table — this indicates a data join issue.

# --- PLACEHOLDER: Opus/Copilot drafts implementation below ---

---
## Step 7: Compute Scenario Demand Profile — Allegheny County (Test Case)
---

Applies the adopter/non-adopter mask to produce a scenario hourly demand profile.

**Logic (per building, per hour):**
- If `bldg_id` is in `adopter_ids` → use `retrofit_kwh` (post-HP electricity)
- Otherwise → use `baseline_kwh` (existing equipment electricity)

**Two profiles produced:**
1. **100% adoption:** all filtered buildings adopt → all use `retrofit_kwh`
2. **Constrained adoption:** Tier 1 + Tier 2 adopt; Tier 3 + Tier 4 use baseline

**Peak load change** = `max(scenario_profile_kw)` − `max(baseline_profile_kw)`

**Units note:** BuildStockQuery returns kWh per hour. Since each interval is 1 hour,
kWh = kW for that hour. Multiply by sampling weight (~240) to convert from simulated
units to real-world MW equivalents.

In [5]:
# CONTEXT: Computing county-level scenario demand profiles for Allegheny County, PA.
# We have baseline and upgrade timeseries DataFrames from Steps 5–6.
# The adopter mask determines which buildings use baseline vs retrofit consumption.
#
# TASK: Implement `compute_county_scenario_profile()` that:
#   1. Merges `df_ts_baseline_allegheny` and `df_ts_upgrade_allegheny` on [bldg_id, hour].
#   2. Accepts `adopter_bldg_ids` (List[int]) as parameter.
#   3. For each building-hour: assigns retrofit_kwh if adopter, else baseline_kwh.
#   4. Aggregates across all buildings → hourly scenario demand profile (8760 values).
#   5. Applies sampling weight to convert to MW (confirm weight value from EUSS metadata).
#   6. Returns a DataFrame with columns: [hour, baseline_mw, scenario_mw, delta_mw].
#
# Then call this function twice:
#   (a) adopter_bldg_ids = adopter_ids_by_county["42003"]["all_filtered"]   (100% adoption)
#   (b) adopter_bldg_ids = adopter_ids_by_county["42003"]["constrained"]    (Tier 1+2 only)
#
# Print peak hour and peak load change for both scenarios.
#
# INPUTS:
#   - `df_ts_baseline_allegheny` (pd.DataFrame): [bldg_id, hour, baseline_kwh]
#   - `df_ts_upgrade_allegheny` (pd.DataFrame): [bldg_id, hour, retrofit_kwh]
#   - `adopter_ids_by_county` (dict): from Step 4
# OUTPUTS:
#   - `df_profile_100pct` (pd.DataFrame): [hour, baseline_mw, scenario_mw, delta_mw]
#   - `df_profile_constrained` (pd.DataFrame): same schema, Tier 1+2 adopters only
#   - `peak_results_allegheny` (dict): peak hour, baseline peak MW, scenario peak MW, delta MW
# CONSTRAINTS: Type hints required. Google/NumPy docstring on the function.
#              Verify 8760 rows in output — raise ValueError if not.

def compute_county_scenario_profile(
    df_baseline: "pd.DataFrame",
    df_upgrade: "pd.DataFrame",
    adopter_bldg_ids: "list[int]",
    sampling_weight: float = 240.0,
) -> "pd.DataFrame":
    """
    # --- PLACEHOLDER: Opus/Copilot completes this function ---
    """
    raise NotImplementedError("Step 7 placeholder — implement with Copilot/Opus")

---
## Step 8: Validate — Compare Aggregated Profile Against EUSS Peak Load Columns
---

EUSS annual/metadata files include individual household peak load values (kW).
These provide a within-dataset sanity check before comparing to external benchmarks.

**Validation approach:**
- Sum the EUSS individual peak load values for Allegheny County buildings → county peak (naive sum)
- Compare to the profile-derived peak from Step 7 (baseline, 100% adoption)
- These will not be identical (naive sum ≠ coincident peak), but should be same order of magnitude

**External benchmark (after internal check passes):**
- Compare PA statewide peak load change to Maxim et al. (2024) 2018 vs. 2050 scenarios
- Reference OEDI URL from notebook stub:
  `https://data.openei.org/s3_viewer?bucket=oedi-data-lake&prefix=nrel-pds-building-stock%2Fend-use-load-profiles-for-us-building-stock%2F2022%2Fresstock_amy2018_release_1.1%2F`

**Flag:** If profile-derived peak differs from EUSS column sum by more than 20%, investigate
before scaling to national loop.

In [ ]:
# CONTEXT: Validating our aggregated timeseries-derived peak load against the EUSS
# annual metadata file's built-in peak load columns for Allegheny County, PA.
# df_baseline (the annual/metadata CSV) is already loaded from Step 1 of the notebook.
#
# TASK:
#   1. Identify the peak load column(s) in `df_baseline` (annual CSV). 
#      Look for columns containing 'peak' or 'max' in the column name — print candidates.
#   2. Filter df_baseline to Allegheny County buildings (use county_geo_df mapping).
#   3. Sum the individual peak load values → `naive_county_peak_kw`.
#      (Naive sum is an upper bound — assumes all buildings peak simultaneously.)
#   4. Compare to `peak_results_allegheny["baseline_peak_mw"] * 1000` (convert to kW).
#   5. Compute ratio: naive_sum / profile_derived_peak. Print result with interpretation.
#   6. Flag with a warning if ratio is outside [0.8, 5.0] (rough sanity bounds).
#
# INPUTS:
#   - `df_baseline` (pd.DataFrame): EUSS annual metadata CSV, loaded in Step 1
#   - `county_geo_df` (pd.DataFrame): county mapping from Step 3
#   - `peak_results_allegheny` (dict): peak load results from Step 7
# OUTPUTS: Printed validation summary. No new DataFrames required.
# CONSTRAINTS: Type hints on any helper. If peak column name is uncertain,
#              print all column names containing 'peak', 'max', or 'load'.

# --- PLACEHOLDER: Opus/Copilot drafts implementation below ---

---
## Step 9: National Loop — County-Level Peak Load Table
---

⚠️ **Run Step 7 validation on Allegheny County BEFORE running this step.**

Scales the Allegheny County pipeline across all counties present in `adopter_ids_by_county`.

**Design decisions to resolve before running:**
1. **Query batching:** Query all buildings for a county in one Athena call, or batch by
   state first (reduces number of Athena connections)?
2. **Aggregation location:** Push the hourly sum to Athena SQL (cheaper, faster) or pull
   building-level data and aggregate in pandas (more flexible for adopter mask)?
3. **Checkpoint saving:** Save intermediate results per state to disk in case the loop fails
   mid-run (strongly recommended for a national run).

**Expected output:** `df_peak_results_national` with one row per county:
`[fips, county_name, state, n_adopters_constrained, n_all_filtered,
  baseline_peak_mw, scenario_100pct_peak_mw, scenario_constrained_peak_mw,
  delta_100pct_mw, delta_constrained_mw, peak_hour_100pct, peak_hour_constrained]`

In [ ]:
# CONTEXT: Scaling the county-level peak load pipeline (Steps 5–7) to all counties
# in the TARE model results. ResStock EUSS 2022.1.1 via AWS Athena / BuildStockQuery.
# Python 3.11, conda env cmu-tare-model. Allegheny County (FIPS 42003) validated in Step 7.
#
# TASK: Implement `run_national_peak_load_loop()` that:
#   1. Iterates over all FIPS keys in `adopter_ids_by_county`.
#   2. For each county: queries baseline + upgrade timeseries (reuse logic from Steps 5–6),
#      calls `compute_county_scenario_profile()` for both adoption scenarios.
#   3. Stores peak load results in a running list → converts to DataFrame at end.
#   4. Saves a checkpoint file per state (e.g., `peak_results_PA.csv`) so progress is
#      not lost if the loop fails.
#   5. Prints progress: "County X of N | FIPS {fips} | {county_name}, {state} | ✓"
#   6. Returns `df_peak_results_national` (pd.DataFrame) with schema described above.
#
# INPUTS:
#   - `rsq` (ResStockQuery): initialized BuildStockQuery object
#   - `adopter_ids_by_county` (dict): from Step 4
#   - `county_geo_df` (pd.DataFrame): from Step 3
#   - `primary_mp` (int): selected measure package
#   - `PROJECT_ROOT` (str): from config, for checkpoint file paths
# OUTPUTS:
#   - `df_peak_results_national` (pd.DataFrame): county-level peak load results
# CONSTRAINTS: Type hints. Google/NumPy docstring. Wrap each county query in try/except
#              — log failures to a `failed_counties` list and continue.
#              Do not run without Step 7 validation passing first.

def run_national_peak_load_loop(
    rsq: object,
    adopter_ids_by_county: "dict[str, dict[str, list[int]]]",
    county_geo_df: "pd.DataFrame",
    primary_mp: int,
    project_root: str,
    sampling_weight: float = 240.0,
) -> "pd.DataFrame":
    """
    # --- PLACEHOLDER: Opus/Copilot completes this function ---
    """
    raise NotImplementedError("Step 9 placeholder — implement with Copilot/Opus")

---
## Step 10: Export Results for Paper Figures
---

Exports the national county-level peak load table for use in:
- **Figure XX (paper Section 3.6):** County choropleth of peak load change under
  100% adoption and economically-constrained adoption, by technology scenario (MP3/MP4)
- **Allegheny County case study panel:** Bar chart comparing baseline vs scenario peak
  across all technology scenarios

Files exported:
- `peak_load_results_MP{mp}_national.csv` — full national county table
- `peak_load_results_MP{mp}_allegheny.csv` — Allegheny County only (for case study)

In [ ]:
# CONTEXT: Exporting national county-level peak load results for paper figures.
# Results are in `df_peak_results_national` from Step 9.
#
# TASK:
#   1. Define OUTPUT_DIR using PROJECT_ROOT (create directory if it doesn't exist).
#   2. Save `df_peak_results_national` as CSV with filename including MP number and date.
#   3. Filter to Allegheny County (FIPS 42003) → save as separate CSV.
#   4. Print file paths and row counts for both exports.
#   5. Print a summary table: top 10 counties by peak load delta (100% adoption scenario).
#
# INPUTS:
#   - `df_peak_results_national` (pd.DataFrame): from Step 9
#   - `primary_mp` (int): for filename
#   - `PROJECT_ROOT` (str): from config
# OUTPUTS: Two CSV files written to disk. Printed confirmation.
# CONSTRAINTS: Use pathlib.Path throughout. Include ISO date in filename.
#              Do not overwrite existing files — append a suffix if file exists.

# --- PLACEHOLDER: Opus/Copilot drafts implementation below ---

---
## 🔖 Handoff Notes for Claude Opus
---

**What is complete (do not modify):**
- Steps 0–0c: TARE data loading, MP selection — working, tested
- Step 1: EUSS annual CSV loading + 3-stage filter — working, tested

**What needs implementation (in order):**
1. Step 1 (NEW): BuildStockQuery install + AWS credential check
2. Step 2: OEDI Athena schema discovery — **must resolve column names before any other step**
3. Step 3: County geography mapping (FIPS/GISJOIN → shapefile)
4. Step 4: Adopter ID extraction from TARE results
5. Steps 5–6: Single-county test queries (Allegheny County first, always)
6. Step 7: Scenario demand profile + peak calculation
7. Step 8: Validation against EUSS built-in peak values
8. Step 9: National loop (only after Step 8 passes)
9. Step 10: Export

**Critical constraint:** ResStock 2022.1.1 (EUSS) only. Do not reference 2025.1.

**Primary test case:** Allegheny County, PA — FIPS 42003.

**Code standards:** Type hints on all functions. Google/NumPy docstrings. Fail fast.

**If Athena table names are unknown:** Check the OEDI S3 browser first:
`https://data.openei.org/s3_viewer?bucket=oedi-data-lake&prefix=nrel-pds-building-stock%2Fend-use-load-profiles-for-us-building-stock%2F2022%2Fresstock_amy2018_release_1.1%2F`
Then check the BuildStockQuery wiki for the correct initialization parameters.

---
## Geospatial Visualization
---

In [ ]:
gdf_conus = None
gdf_alaska = None

try:
    gdf_states_raw = gpd.read_file(SHAPEFILE_PATH)
    _, gdf_conus, gdf_alaska = prepare_state_geodataframe(gdf_states_raw, df_spark, merge_col='state')
    print(f"✓ Geodataframe prepared: CONUS={len(gdf_conus)}, AK={len(gdf_alaska)}")
except Exception as e:
    print(f"⚠ Shapefile not loaded: {e} — skipping maps")

In [ ]:
# Demand change map (diverging)
if gdf_conus is not None and gdf_alaska is not None:
    _, gdf_demand_conus, gdf_demand_alaska = prepare_state_geodataframe(
        gdf_states_raw, df_demand_state, merge_col='state'
    )
    create_choropleth_map(
        gdf_demand_conus, gdf_demand_alaska,
        column='elec_change_gwh',
        title='Electricity Demand Change Under 100% HP Adoption by State (2022)',
        cbar_label='Electricity Demand Change (GWh)\n(positive = more grid electricity needed)',
        output_path=os.path.join(PROJECT_ROOT, "state_elec_demand_change_map_2022.png"),
        cmap='coolwarm', show_plot=True,
    )
    print("✓ Demand map generated")
else:
    print("⚠ Maps skipped")

---
## Display Results
---

In [ ]:
# ============================================================================
# DISPLAY: DEMAND CHANGE
# ============================================================================

# print(f"\n===== DEMAND CHANGE (MP{primary_mp}, GWh, all fuels, 100% adoption) =====\n")
# display(df_demand_state[['state', 'home_count', 'elec_change_gwh',
#                           'pct_elec_demand_change', 'site_energy_change_gwh',
#                           'pct_site_energy_change']])

# print(f"\n✓ DISPLAY COMPLETE")